---
# Combinación de conjuntos de datos: fusionar y unir
---

Una característica esencial que ofrece Pandas son sus operaciones de unión y fusión en memoria de alto rendimiento.<br>
Si alguna vez ha trabajado con bases de datos, debería estar familiarizado con este tipo de interacción de datos. <br>
La interfaz principal para esto es la función ``pd.merge``, y veremos algunos ejemplos de cómo esto puede funcionar en la práctica.

Por conveniencia, comenzaremos redefiniendo la funcionalidad ``display()`` de la sección anterior , esto lo haremos al solo efecto de poder mostrar visualmente varios conjuntos de datos al mismo tiempo:

In [ ]:
import pandas as pd
import numpy as np

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)


# <span style="color:orange"> 1. Álgebra relacional <br>

El comportamiento implementado en ``pd.merge()`` es un subconjunto de lo que se conoce como *álgebra relacional*, que es un conjunto formal de reglas para manipular datos relacionales y forma la base conceptual de las operaciones disponibles en la mayoría de las bases de datos. <br>
La fortaleza del enfoque del álgebra relacional es que propone varias operaciones primitivas, que se convierten en los componentes básicos de operaciones más complicadas en cualquier conjunto de datos. <br>
Con este léxico de operaciones fundamentales implementado eficientemente en una base de datos u otro programa, se puede realizar una amplia gama de operaciones compuestas bastante complicadas.

Pandas implementa varios de estos bloques de construcción fundamentales en la función ``pd.merge()`` y el método relacionado ``join()`` de ``Series`` y ``Dataframe``.<br>
Como veremos, estos le permiten vincular de manera eficiente datos de diferentes fuentes.

## <span style="color:orange"> 2. Categorías de uniones </span>

La función ``pd.merge()`` implementa varios tipos de uniones: las uniones *uno a uno*, *muchos a uno* y *muchos a muchos*.<br>
Se accede a los tres tipos de combinaciones mediante una llamada idéntica a la interfaz ``pd.merge()``.<br> 
El tipo de unión realizada depende de la forma de los datos de entrada.<br>
Aquí mostraremos ejemplos simples de los tres tipos de fusiones y analizaremos las opciones detalladas más adelante.

### <span style="color:orange"> 2.1. Uniones uno a uno </span>

Quizás el tipo más simple de expresión de fusión es la unión uno a uno, que en muchos aspectos es muy similar a la concatenación de columnas.<br>
Como ejemplo concreto, considere los siguientes dos ``DataFrames`` que contienen información sobre varios empleados de una empresa:

In [ ]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})
df1

In [ ]:
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})
df2

In [ ]:
display('df1', 'df2')

Para combinar esta información en un único ``DataFrame``, podemos usar la función ``pd.merge()``:

In [ ]:
df3=pd.merge(df1,df2)
df3

La función ``pd.merge()`` reconoce que cada ``DataFrame`` tiene una columna de "employee" y se une automáticamente usando esta columna como clave.
El resultado de la fusión es un nuevo ``DataFrame`` que combina la información de las dos entradas.<br>
Tenga en cuenta que el orden de las entradas en cada columna no se mantiene necesariamente: en este caso, el orden de la columna "employee" difiere entre ``df1`` y ``df2``, y ``pd.merge()``La función tiene en cuenta esto correctamente. <br>
Además, tenga en cuenta que la fusión en general descarta el índice, excepto en el caso especial de fusiones por índice (consulte las palabras clave ``left_index`` y ``right_index``, que se analizan momentáneamente).

### <span style="color:orange"> 2.2. Join de muchos a uno
Las combinaciones de muchos a uno son combinaciones en las que una de las dos columnas clave contiene entradas duplicadas. <br>
Para el caso de muchos a uno, el ``DataFrame`` resultante conservará esas entradas duplicadas según corresponda.<br>
Considere el siguiente ejemplo de una unión de muchos a uno:

In [ ]:
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})

df4


In [ ]:
display('df3','df4', 'pd.merge(df3,df4)')

In [ ]:
pd.merge(df3,df4)

El ``DataFrame`` resultante tiene una columna adicional con la información del "supervisor", donde la información se repite en una o más ubicaciones según lo requieran las entradas.

### <span style="color:orange"> 2.3. Joins de muchos a muchos
Las uniones de muchos a muchos son un poco confusas conceptualmente, pero aun así están bien definidas. <br>
Si la columna clave en la matriz izquierda y derecha contiene duplicados, entonces el resultado es una combinación de muchos a muchos.<br>
Quizás esto quede más claro con un ejemplo concreto.<br>
Considere lo siguiente, donde tenemos un ``DataFrame`` que muestra una o más habilidades asociadas con un grupo en particular.<br>
Al realizar una unión de muchos a muchos, podemos recuperar las habilidades asociadas a cualquier persona individual:

In [ ]:
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting',
                              'Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'coding', 'linux',
                               'spreadsheets', 'organization']})

df5


In [ ]:
display('df1','df5', 'pd.merge(df1 , df5)')

Estos tres tipos de uniones se pueden utilizar con otras herramientas de Pandas para implementar una amplia gama de funciones.<br>
Pero en la práctica, los conjuntos de datos rara vez son tan limpios como el que estamos trabajando aquí.<br>
En la siguiente sección consideraremos algunas de las opciones proporcionadas por ``pd.merge()`` que le permiten ajustar cómo funcionan las operaciones de unión.

## <span style="color:orange"> 3. Especificación de Merge Key
Ya hemos visto el comportamiento predeterminado de ``pd.merge()``: busca uno o más nombres de columnas coincidentes entre las dos entradas y lo usa como clave.<br>
Sin embargo, a menudo los nombres de las columnas no coinciden tan bien y ``pd.merge()`` proporciona una variedad de opciones para manejar esto.

### <span style="color:orange"> 3.1. La palabra clave ``on``

Lo más simple es que puedes especificar explícitamente el nombre de la columna clave usando la palabra clave ``on``, que toma un nombre de columna o una lista de nombres de columnas:

In [ ]:
display('df1','df2', "pd.merge(df1 , df2 , on='employee')")

Esta opción sólo funciona si tanto el ``DataFrame`` izquierdo como el derecho tienen el nombre de columna especificado.

### <span style="color:orange"> 3.2. Las palabras clave ``left_on`` y ``right_on``

En ocasiones, es posible que desee fusionar dos conjuntos de datos con nombres de columnas diferentes; por ejemplo, es posible que tengamos un conjunto de datos en el que el nombre del empleado esté etiquetado como "nombre" en lugar de "empleado".<br>
En este caso, podemos usar las palabras clave ``left_on`` y ``right_on`` para especificar los nombres de las dos columnas:

In [ ]:
df3 = pd.DataFrame({'name': ['Bob', 'Lisa','Jake', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})

df3


In [ ]:
display('df1','df3', "pd.merge(df1, df3, left_on='employee', right_on='name')")

In [ ]:
#display('df1','df3', "pd.merge(df1, df3)") # si no hay columnas en comun no hace el merge

El resultado tiene una columna redundante que podemos eliminar si lo deseamos, por ejemplo, usando el método ``drop()`` de ``DataFrame``s:

In [ ]:
pd.merge(df1, df3, left_on='employee', right_on='name').drop('name', axis=1)

### <span style="color:orange"> 3.3. Las palabras clave ``left_index`` y ``right_index``

A veces, en lugar de fusionar en una columna, le gustaría fusionar en un índice.<br>
Por ejemplo, sus datos podrían verse así:

In [ ]:
df1_set_index = df1.set_index('employee')
df1_set_index


In [ ]:
df2_set_index = df2.set_index('employee')
df2_set_index

In [ ]:
df3_set_index= df3.set_index('name')
df3_set_index

Puedes usar el índice como clave para fusionar especificando los indicadores ``left_index`` y/o ``right_index`` en ``pd.merge()``:

In [ ]:
display('df1_set_index', 'df2_set_index' , "pd.merge(df1_set_index,df2_set_index, left_index=True,right_index=True)")

In [ ]:
display('df1_set_index', 'df3_set_index' , "pd.merge(df1_set_index,df3_set_index, left_index=True,right_index=True)")

Para mayor comodidad, los ``DataFrame`` implementan el método ``join()``, que realiza una fusión que por defecto se une en índices:

In [ ]:
display('df1_set_index','df2_set_index', 'df1_set_index.join(df2_set_index)')

Si desea mezclar índices y columnas, puede combinar ``left_index`` con ``right_on`` o ``left_on`` con ``right_index`` para obtener el comportamiento deseado:

In [ ]:
display('df1_set_index','df3'," pd.merge(df1_set_index ,df3, left_index=True, right_on='name')")

## <span style="color:orange"> 4. Especificación de aritmética de conjuntos para joins

En todos los ejemplos anteriores hemos pasado por alto una consideración importante al realizar una unión: el tipo de aritmética de conjuntos utilizada en la unión.<br>
Esto aparece cuando aparece un valor en una columna clave pero no en la otra. Considere este ejemplo:


<div style="text-align: center;">
  <img src="Merges.png" width="400">
</div>


In [ ]:
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']},
                   columns=['name', 'food'])

df6

In [ ]:
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']},
                   columns=['name', 'drink'])
df7

In [ ]:
display('df6','df7', "pd.merge(df6,df7)")

Aquí hemos fusionado dos conjuntos de datos que tienen una sola entrada de "nombre" en común: María.<br>
De forma predeterminada, el resultado contiene la *intersección* de los dos conjuntos de entradas; esto es lo que se conoce como *inner join*.<br>
Podemos especificar esto explícitamente usando la palabra clave ``how``, que por defecto es ``"inner"``:

In [ ]:
pd.merge(df6,df7,how='inner')

Otras opciones para la palabra clave  ``how`` keyword son ``'outer'``, ``'left'``, y ``'right'``.
Un *outer join* devuelve una unión sobre la unión de las columnas de entrada y completa todos los valores faltantes con NA:

### Veamos que pasa si aplicamos un metodo outer join

In [ ]:
display('df6','df7', "pd.merge(df6 ,df7 , how= 'outer')")

El *left join* y *right join* return une las entradas izquierda y derecha, respectivamente.<br>

### Veamos que pasa si aplicamos un metodo left join

In [ ]:
display('df6','df7', "pd.merge(df6, df7 , how='left')")

Las filas de salida ahora corresponden a las entradas en la entrada izquierda. Usando ``how='right'`` funciona de manera similar.

Todas estas opciones se pueden aplicar directamente a cualquiera de los tipos de unión anteriores.

### Veamos que pasa si aplicamos un metodo right join

In [ ]:
display('df6','df7', "pd.merge(df6, df7, how='right')")

y si invertimos los df y vemos que pasa:

In [ ]:
display('df6','df7', "pd.merge(df6, df7, how='left')")

In [ ]:
display('df6','df7', "pd.merge(df7, df6, how='left')")

## <span style="color:orange"> 5. Nombres de columnas superpuestas: la palabra clave (Keyword) ``suffixes`` 
Finalmente, puede terminar en un caso en el que sus dos ``DataFrame``s de entrada tengan nombres de columnas en conflicto.<br>
Considere este ejemplo:

In [ ]:
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})
df8



In [ ]:
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [3, 1, 4, 2]})
df9

In [ ]:
display('df8','df9', "pd.merge(df8,df9 ,on='name')")

Debido a que la salida tendría dos nombres de columnas en conflicto, la función de combinación agrega automáticamente un sufijo ``_x`` o ``_y`` para que las columnas de salida sean únicas. <br>
Si estos valores predeterminados no son apropiados, es posible especificar un sufijo personalizado usando la palabra clave ``suffixes``:

In [ ]:
display('df8','df9', 'pd.merge(df8 , df9 , on="name" , suffixes=["_L" , "_R"] )')

Estos sufijos funcionan en cualquiera de los patrones de unión posibles y también funcionan si hay varias columnas superpuestas.

## <span style="color:orange"> 6. Ejemplo: datos de los estados de EE. UU.

Las operaciones de fusión y unión surgen con mayor frecuencia cuando se combinan datos de diferentes fuentes.<br>
Aquí consideraremos un ejemplo de algunos datos sobre los estados de EE. UU. y sus poblaciones.<br>
Los archivos de datos se pueden encontrar en http://github.com/jakevdp/data-USstates/:

In [ ]:
# Obtenemos mediante url las direcciones de los dataset:
url1 = 'https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-population.csv'
url2 = 'https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-areas.csv'
url3 = 'https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-abbrevs.csv'

Echemos un vistazo a los tres conjuntos de datos, usando la función ``read_csv()`` de Pandas:

Dada esta información, supongamos que queremos calcular un resultado relativamente sencillo: clasificar los estados y territorios de EE. UU. según su densidad de población en 2010.<br>
Claramente tenemos los datos aquí para encontrar este resultado, pero tendremos que combinar los conjuntos de datos para encontrar el resultado.

Comenzaremos con una fusión de muchos a uno que nos dará el nombre completo del estado dentro de la población ``DataFrame``.<br>
Queremos fusionarnos en función de la columna ``estado/región`` de ``pop`` y la columna ``abreviatura`` de ``abbrevs``.<br>
Usaremos ``how='outer'`` para asegurarnos de que no se desperdicie ningún dato debido a etiquetas que no coinciden.

In [ ]:
# Merge pop con abbrevs basado en la columna 'state/region' en pop y 'abbreviation' en abbrevs



In [ ]:
# Eliminar la columna 'abbreviation' ya que es redundante



Verifiquemos nuevamente si hubo discrepancias aquí, lo cual podemos hacer buscando filas con valores nulos:

Parte de la información de la ``población`` es nula; ¡Averigüemos cuáles son!

Parece que todos los valores nulos de población son de Puerto Rico anteriores al año 2000; Es probable que esto se deba a que estos datos no están disponibles en la fuente original.

Más importante aún, vemos también que algunas de las nuevas entradas de ``state`` también son nulas, lo que significa que no había ninguna entrada correspondiente en la clave ``abrevs``.<br>
Averigüemos qué regiones carecen de esta coincidencia:

Podemos inferir rápidamente el problema: nuestros datos de población incluyen entradas para Puerto Rico (PR) y los Estados Unidos en su conjunto (EE.UU.), mientras que estas entradas no aparecen en la clave de abreviatura estatal.<br>
Podemos solucionarlos rápidamente completando las entradas apropiadas:

No más valores nulos en la columna ``estado``: ¡ya estamos listos!

Ahora podemos fusionar el resultado con los datos del área usando un procedimiento similar.<br>
Al examinar nuestros resultados, querremos unirnos en la columna ``estado`` en ambos:

Nuevamente, revisemos los valores nulos para ver si hubo alguna discrepancia:

Hay valores nulos en la columna ``área``; Podemos echar un vistazo para ver qué regiones se ignoraron aquí:

Vemos que nuestras ``áreas`` ``DataFrame`` no contiene el área de Estados Unidos en su conjunto.<br>
Podríamos insertar el valor apropiado (usando la suma de todas las áreas estatales, por ejemplo), pero en este caso simplemente eliminaremos los valores nulos porque la densidad de población de todo Estados Unidos no es relevante para nuestra discusión actual:

Ahora tenemos todos los datos que necesitamos. Para responder a la pregunta de interés, seleccionemos primero la parte de los datos correspondiente al año 2010 y la población total.

Ahora veamos los Estados con una poblacion mayor a 5 Millones